## Pre-process data — Phytoplankton

Crops every classified particle out of its FlowCam mosaic collage and writes one
`.tif` per particle into `OUTPUT_DIR/<class>/`, ready for `ImageDataset`.

Adapted from `HierNet/Zooplankton-MNR/pre_process_data.ipynb`. Two things differ
for this dataset:

1. **Matching is done on the CSV `Image File` column, not on filenames.** The CSV
   file names do not line up with the tif folder names (`4X300` vs `4X300um` vs
   `4x3000um`, `Lousia`/`Louisa`, `Sagnanagons`, `rep1`/`Rep1`, a trailing
   `_redo`, …), so a filename-stem match would drop most of the data.
2. **`Particle ID` is not unique within a mosaic** here, so the crop's top-left
   `(x, y)` is added to each output filename to keep names collision-free.

Re-running is safe: any output file that already exists is left untouched and
counted as *skipped* (same idempotent behaviour as the Zooplankton notebook).

In [1]:
import os
import glob
import collections

import pandas as pd
from PIL import Image

### Expected input layout

```
/data/Hier-Phytoplankton/
├── 2025 Quetico/
│   ├── 2025 quetico csv/
│   │   ├── 20250522_4X300_Mack_rep1_PartialAI_data.csv
│   │   └── ...
│   ├── 20250522_4X300um_Mack_rep1/
│   │   ├── 20250522_4X300um_Mack_rep1_000001.tif   <- mosaic collage
│   │   ├── cal_image_000001.tif                    <- calibration frame (ignored)
│   │   └── *.cla / *.ctx / *.edg / *.lst / *.txt   <- FlowCam metadata (ignored)
│   └── ...
└── Cradles/
    ├── Cradles csv/
    │   └── ...
    └── ...
```

Each mosaic `.tif` is a collage of many particle thumbnails; the matching CSV row
gives a particle's `Class` and its bounding box (`Image X`, `Image Y`,
`Image Width`, `Image Height`) within that collage.

In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_ROOT = '/data/Hier-Phytoplankton'

# Raw collection folders. Each holds per-sample-run subfolders of mosaic ".tif"
# collages plus one "*csv*" subfolder of FlowCam particle-export CSVs.
COLLECTIONS = ['2025 Quetico', 'Cradles']

# Cropped single-particle images go to OUTPUT_DIR/<class>/.
# Set OUTPUT_DIR = DATA_ROOT to put the class folders at the dataset root
# (matching the /data/Hier-Zooplankton layout) instead of under Processed_data/.
OUTPUT_DIR = os.path.join(DATA_ROOT, 'Processed_data')

# Columns every CSV row needs for a crop.
CROP_COLS = ['Particle ID', 'Class', 'Image File',
             'Image X', 'Image Y', 'Image Width', 'Image Height']

# ---------------------------------------------------------------------------
# Index every mosaic .tif by basename.
#
# The "Image File" column inside each CSV always names the mosaic tif exactly,
# so we match on that against this basename -> path index (CSV *filenames* are
# not a reliable key -- see the notebook header).
#
# "cal_image_*.tif" are per-run calibration frames, never referenced by a CSV
# row; they are skipped.
# ---------------------------------------------------------------------------
tif_index = {}
_seen = collections.defaultdict(list)
for collection in COLLECTIONS:
    collection_path = os.path.join(DATA_ROOT, collection)
    for entry in sorted(os.listdir(collection_path)):
        sample_dir = os.path.join(collection_path, entry)
        if not os.path.isdir(sample_dir) or 'csv' in entry.lower():
            continue
        for tif in glob.glob(os.path.join(sample_dir, '*.tif')):
            name = os.path.basename(tif)
            if name.startswith('cal_image'):
                continue
            _seen[name].append(tif)
            tif_index[name] = tif
tif_collisions = {n: p for n, p in _seen.items() if len(p) > 1}

# Collect CSV export files from each collection's "*csv*" subfolder.
csv_files = []
for collection in COLLECTIONS:
    collection_path = os.path.join(DATA_ROOT, collection)
    for d in sorted(os.listdir(collection_path)):
        d_path = os.path.join(collection_path, d)
        if os.path.isdir(d_path) and 'csv' in d.lower():
            csv_files.extend(sorted(glob.glob(os.path.join(d_path, '*.csv'))))

print(f'Mosaic tifs indexed : {len(tif_index)}')
print(f'CSV export files    : {len(csv_files)}')
print(f'Output directory    : {OUTPUT_DIR}')
if tif_collisions:
    print(f'\nWARNING: {len(tif_collisions)} mosaic basename(s) live in >1 sample '
          f'folder; the last indexed wins:')
    for name, paths in tif_collisions.items():
        print(f'  {name}: {paths}')

Mosaic tifs indexed : 114
CSV export files    : 74
Output directory    : /data/Hier-Phytoplankton/Processed_data


### Sanity checks

Run before cropping to see the class breakdown and catch missing / unmatched
data. Nothing is written to disk here.

In [3]:
n_rows = 0
class_counts = collections.Counter()
rows_missing_tif = collections.Counter()   # Image File -> row count
rows_missing_coords = 0
csv_bad_cols = []
referenced_mosaics = set()

for csv in csv_files:
    df = pd.read_csv(csv)
    missing = [c for c in CROP_COLS if c not in df.columns]
    if missing:
        csv_bad_cols.append((os.path.basename(csv), missing))
        continue
    n_rows += len(df)
    class_counts.update(df['Class'].dropna().astype(str))
    referenced_mosaics.update(df['Image File'].dropna())
    coord_na = df[['Image X', 'Image Y', 'Image Width', 'Image Height']].isna().any(axis=1)
    rows_missing_coords += int(coord_na.sum())
    for img in df.loc[~df['Image File'].isin(tif_index), 'Image File'].dropna():
        rows_missing_tif[img] += 1

# Sample folders that hold mosaics but that no CSV references at all.
folders_without_csv = []
for collection in COLLECTIONS:
    collection_path = os.path.join(DATA_ROOT, collection)
    for entry in sorted(os.listdir(collection_path)):
        sample_dir = os.path.join(collection_path, entry)
        if not os.path.isdir(sample_dir) or 'csv' in entry.lower():
            continue
        mosaics = [os.path.basename(t) for t in glob.glob(os.path.join(sample_dir, '*.tif'))
                   if not os.path.basename(t).startswith('cal_image')]
        if mosaics and not any(m in referenced_mosaics for m in mosaics):
            folders_without_csv.append(os.path.join(collection, entry))

print(f'Total particle rows across {len(csv_files)} CSVs : {n_rows}')
print(f'Distinct classes                              : {len(class_counts)}')
print(f'Rows with missing crop coordinates            : {rows_missing_coords}')
print(f'Rows whose "Image File" is not on disk        : {sum(rows_missing_tif.values())}')
print()
print('Class breakdown (verbatim from the CSVs):')
for cls, n in class_counts.most_common():
    print(f'  {n:6d}  {cls}')

if csv_bad_cols:
    print('\nWARNING: CSVs missing required columns (will be skipped):')
    for name, miss in csv_bad_cols:
        print(f'  {name}: {miss}')
if folders_without_csv:
    print('\nWARNING: sample folders with mosaics but no CSV coverage '
          '(their particles will NOT be cropped):')
    for f in folders_without_csv:
        print(f'  {f}')
if rows_missing_tif:
    print(f'\nWARNING: {len(rows_missing_tif)} "Image File" value(s) referenced by a '
          f'CSV are missing on disk:')
    for name, n in rows_missing_tif.most_common(20):
        print(f'  {n:4d}  {name}')

Total particle rows across 74 CSVs : 9920
Distinct classes                              : 24
Rows with missing crop coordinates            : 0
Rows whose "Image File" is not on disk        : 0

Class breakdown (verbatim from the CSVs):
    3250  Diatom
    3179  Not Phytoplankton
    1209  Dinobryon
     570  Zooplankton
     396  Phytoplankton
     355  SP_Colony
     306  Unclear_Filament
     157  Unclear
     133  Dinoflagellate
     109  Bubble
      44  Colony
      43  Empty Cells
      39  MultiClass
      25  Blurry
      20  Empty Cell
      18  Unknown sp1
      15  Unknown
      15  DarkClump
      10  Filament_Likely
       9  Duplicate
       6  Ciliate
       5  Duplicates
       4  Too blurry
       3  EL_Colony

  Cradles/20250820_4x300um_Baby_Rep3


### Optional relabelling / filtering

Classes are extracted **verbatim** by default. The raw label set has near-dupes
(`Empty Cell` / `Empty Cells`, `Duplicate` / `Duplicates`) and junk categories
(`Blurry`, `Too blurry`, `MultiClass`, `DarkClump`, …). Fill in the two
containers below to merge or drop them; leave them empty to keep every class
exactly as recorded.

In [5]:
# Applied in the crop loop below. Examples (commented out):
CLASS_RENAME = {
    # 'Empty Cell': 'Empty Cells',
    # 'Duplicate': 'Duplicates',
    # 'Bubble': 'Bubbles',
}
CLASS_DROP = set([
    # 'MultiClass', 'Blurry', 'Too blurry', 'DarkClump', 'Duplicate', 'Duplicates',
])

### Crop and save

Idempotent — an output file that already exists is left untouched and counted as
*skipped*, so re-running only fills in what is missing.

Output filename: `<mosaic_stem>_p<ParticleID>_x<ImageX>_y<ImageY>.tif`
(`Particle ID` alone is not unique within a mosaic in this dataset).

In [6]:
n_saved = 0
n_skipped = 0
n_dropped = 0
n_missing_tif = 0
missing_tif_names = set()

for csv in csv_files:
    df = pd.read_csv(csv)
    if not set(CROP_COLS).issubset(df.columns):
        continue

    # Normalise column names the same way the Zooplankton notebook does, then
    # give the 'class' column a name that survives itertuples (it is a keyword).
    df = df.rename(columns={c: c.replace(' ', '_').replace('.', '_').lower()
                            for c in df.columns})
    df = df.rename(columns={'class': 'Class'})

    for image_file, group in df.groupby('image_file'):
        tif_path = tif_index.get(image_file)
        if tif_path is None:
            n_missing_tif += len(group)
            missing_tif_names.add(image_file)
            continue

        mosaic = None                                   # opened lazily
        mosaic_stem = os.path.splitext(image_file)[0]

        for row in group.itertuples(index=False):
            cls = CLASS_RENAME.get(str(row.Class), str(row.Class))
            if cls in CLASS_DROP:
                n_dropped += 1
                continue
            if pd.isna(row.image_x) or pd.isna(row.image_y) or \
               pd.isna(row.image_width) or pd.isna(row.image_height):
                n_dropped += 1
                continue

            x, y = int(row.image_x), int(row.image_y)
            w, h = int(row.image_width), int(row.image_height)

            class_dir = os.path.join(OUTPUT_DIR, cls)
            out_path = os.path.join(
                class_dir,
                f'{mosaic_stem}_p{int(row.particle_id)}_x{x}_y{y}.tif',
            )

            if os.path.exists(out_path):                 # already done on a prior run
                n_skipped += 1
                continue

            if mosaic is None:
                mosaic = Image.open(tif_path)

            os.makedirs(class_dir, exist_ok=True)
            mosaic.crop((x, y, x + w, y + h)).save(out_path)
            n_saved += 1

print(f'Newly cropped and saved       : {n_saved}')
print(f'Skipped (already existed)      : {n_skipped}')
print(f'Dropped (CLASS_DROP / no box)  : {n_dropped}')
print(f'Rows with no mosaic on disk    : {n_missing_tif}')
if missing_tif_names:
    print('\nMissing mosaics:')
    for name in sorted(missing_tif_names):
        print(f'  {name}')

Newly cropped and saved       : 9920
Skipped (already existed)      : 0
Dropped (CLASS_DROP / no box)  : 0
Rows with no mosaic on disk    : 0


### Resulting layout

```
<OUTPUT_DIR>/                     # default: /data/Hier-Phytoplankton/Processed_data/
├── Diatom/
│   ├── 20250522_4X300um_Mack_rep1_000001_p1_x0_y0.tif
│   └── ...
├── Dinobryon/
│   └── ...
├── Not Phytoplankton/
│   └── ...
└── ...
```

One `.tif` per particle. Point `ImageDataset` at `<OUTPUT_DIR>` (and pass an
explicit class list so the sibling `2025 Quetico/` and `Cradles/` raw folders are
ignored if `OUTPUT_DIR == DATA_ROOT`).